In [0]:
%pip install -q git+https://github.com/KrajTechne/ColabDesign --no-deps

In [0]:
%restart_python

## Generate a sequence via MPNN and Validate with AlphaFold2

In [0]:
dbutils.widgets.text('path_task_yaml', "")
path_task_yaml = dbutils.widgets.get('path_task_yaml')
#path_task_yaml = "/Volumes/sandbox/karthik/motif_scaffolding/experiments/inputs/gopher_alpha_snake.yaml"
print(path_task_yaml)

In [0]:
dbutils.widgets.text('path_mpnn_af2_yaml', '')
path_mpnn_af2_yaml = dbutils.widgets.get('path_mpnn_af2_yaml')
print(path_mpnn_af2_yaml)
#path_mpnn_af2_yaml = "/Volumes/sandbox/karthik/motif_scaffolding/experiments/rfdiffusion3/designs/rfd3_test_7r2p1/rfd3_test_7r2p1_spec_1/mpnn_af2.yaml"

In [0]:
import yaml
with open(path_mpnn_af2_yaml, 'r') as f:
  mpnn_af2_yaml = yaml.safe_load(f)
mpnn_af2_yaml

In [0]:
import subprocess

#@markdown ProteinMPNN Settings
num_seqs = mpnn_af2_yaml['mpnn']['num_seqs'] #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
#@markdown - `mpnn_sampling_temp` - control diversity of sampled sequences. (higher = more diverse).
mpnn_sampling_temp = mpnn_af2_yaml['mpnn']['mpnn_sampling_temp'] #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5", "1.0"] {type:"raw"}
rm_aa = mpnn_af2_yaml['mpnn']['rm_aa'] #@param {type:"string"}
use_solubleMPNN = mpnn_af2_yaml['mpnn']['use_solubleMPNN'] #@param {type:"boolean"}


#@markdown AlphaFold Settings
#@markdown - soft initialization with desired coordinates, see [paper](https://www.nature.com/articles/s41467-023-38328-5).
initial_guess = mpnn_af2_yaml['alphafold2']['initial_guess'] #@param {type:"boolean"}
#@markdown - for **binder** design, we recommend `initial_guess=True num_recycles=3`
num_recycles = mpnn_af2_yaml['alphafold2']['num_recycles'] #@param ["0", "1", "2", "3", "6", "12"] {type:"raw"}
use_multimer = mpnn_af2_yaml['alphafold2']['use_multimer'] #@param {type:"boolean"}

num_designs = mpnn_af2_yaml['pipeline']['num_designs']
opts = [f"--pdb={mpnn_af2_yaml['pipeline']['pdb']}",
        f"--loc={mpnn_af2_yaml['pipeline']['loc']}",
        f"--contig={mpnn_af2_yaml['pipeline']['contig_str']}",
        f"--copies={mpnn_af2_yaml['pipeline']['copies']}",
        f"--num_seqs={num_seqs}",
        f"--num_recycles={num_recycles}",
        f"--rm_aa={rm_aa}",
        f"--mpnn_sampling_temp={mpnn_sampling_temp}",
        f"--num_designs={num_designs}"]
if initial_guess: opts.append("--initial_guess")
if use_multimer: opts.append("--use_multimer")
if use_solubleMPNN: opts.append("--use_soluble")

# Add in af_params_dir path:
af_params_dir = "/Volumes/sandbox/model_weights/protein_hunter/params/"
opts += [f"--af_params_dir={af_params_dir}"]

cmd = ['python', '/Workspace/Users/karthik.raj@bio-techne.com/ColabDesign/colabdesign/rf/designability_test.py'] + opts
print(" ".join(cmd))
subprocess.run(cmd)

In [0]:
path = mpnn_af2_yaml['pipeline']['loc']
path_rfd_design = mpnn_af2_yaml['pipeline']['pdb']
path_rfd_design_adaptable = '_'.join(path_rfd_design.split('_')[:-1])
path_rfd_design_adaptable

In [0]:
#@title Display best result
import py3Dmol
import ipywidgets as widgets

num_designs = mpnn_af2_yaml['pipeline']['num_designs']
def plot_pdb(num = "best"):
  if num == "best":
    with open(f"{path}/best.pdb","r") as f:
      # REMARK 001 design {m} N {n} RMSD {rmsd}
      info = f.readline().strip('\n').split()
    num = info[3]
  hbondCutoff = 4.0
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
  pdb_str = open(f"{path_rfd_design_adaptable}_{num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  pdb_str = open(f"{path}/best_design{num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})

  view.setStyle({"model":0},{'cartoon':{}}) #: {'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.setStyle({"model":1},{'cartoon':{'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.zoomTo()
  view.show()

if num_designs > 1:
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(
    options=["best"] + [str(k) for k in range(num_designs)],
    value="best",
    description='design:',
  )
  dropdown.observe(on_change)
  output = widgets.Output()
  display(widgets.VBox([dropdown, output]))
  with output:
    plot_pdb(dropdown.value)
else:
  plot_pdb()

### Add Binder Chain RMSD Calculations

In [0]:
import pandas as pd
import os
path = mpnn_af2_yaml['pipeline']['loc']
path_mpnn_csv = f"{path}/mpnn_results.csv"
df_mpnn = pd.read_csv(path_mpnn_csv, index_col=0)
df_mpnn['contig'] = mpnn_af2_yaml['pipeline']['contig_str'].replace(':', ';') 
df_mpnn

In [0]:
import sys
sys.path.append('/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics')
from StrucTools import *
import biotite.structure as struc
path_pdbs = f"{path}/all_pdb"
list_binder_rfd_af2_holo_rmsd = []
for index, row in df_mpnn.iterrows():
    # 1. Define PDB paths of holo complexes for RFDiffusion input to MPNN and the MPNN-derived seqs whose structures predicted via AF2
    pdb_af2_path = path_pdbs + f"/design{row['design']}_n{row['n']}.pdb"
    pdb_rfd_path = f"{path_rfd_design_adaptable}_{row['design']}.pdb"
    print("RFDiffusion Path:", pdb_rfd_path)
    # 2. Extract only the alpha-carbons from the holo complexes
    atom_array_complex_rfd = extract_atom_array(pdb_rfd_path, ca_only = True)
    atom_array_complex_af2 = extract_atom_array(pdb_af2_path, ca_only = True)
    # 3. Create mask for only the binder chain residues
    residue_binder_mask = atom_array_complex_rfd.chain_id == 'B'
    # 4. Spread mask to all atoms in the entire complex
    atom_binder_mask = struc.spread_residue_wise(atom_array_complex_rfd, residue_binder_mask)
    # 5. Superimpose the 2 structures at the atom_binder_mask and calculate the RMSD for the binder chains
    rmsd, _, _ = superimpose_and_calculate_specified_rmsd(atom_array_complex_rfd, atom_array_complex_af2, atom_binder_mask, atom_binder_mask)
    print("RMSD between MPNN-seq AF2 predicted holo structure and original RFDiffusion Holo structure: ", rmsd)
    list_binder_rfd_af2_holo_rmsd.append(rmsd)
df_mpnn['rmsd_holo_binder_rfd_af2'] = list_binder_rfd_af2_holo_rmsd
df_mpnn.to_csv(path_mpnn_csv)
df_mpnn

### Separate: If this is a motif scaffolding problem calculate motif rmsd 

if multiple motifs, calculate rmsd of specific motif of interest

In [0]:
contig_str = mpnn_af2_yaml['pipeline']['contig_str']
contig_target, contig_binder = contig_str.split(':')
if "-" in contig_binder:
    motif_scaffold = True
else:
    motif_scaffold = False
print("Motif Scaffold: ", motif_scaffold)

In [0]:
import yaml

if motif_scaffold:
    with open(path_task_yaml) as file:
        config_task = yaml.safe_load(file)

    path_input_pdb = config_task['path_input_pdb']
    if "motif_chain_id" in config_task:
        desired_motif_chain_id = config_task['motif_chain_id']
    else:
        desired_motif_chain_id = "X"
    print("Path Input PDB: ", path_input_pdb)
    print("Desired Motif Chain ID: ", desired_motif_chain_id)

In [0]:
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/MotifScaffold/utils")
from utils import *
from RFDUtils import *
import biotite.structure.io as bsio

def compute_chain_plddt(path_pdb_design: str, chain_id: str = 'B'):
    """ Compute pLDDT of a chain in a PDB file from Sergey's Workflow """
    struct = bsio.load_structure(path_pdb_design, extra_fields=["b_factor"])
    struct_chain = struct[struct.chain_id == chain_id]
    plddt_chain = struct_chain[struct_chain.atom_name == 'CA'].b_factor.mean()
    return plddt_chain

def calculate_rmsd_aligned_coords(path_pdb_design: str, path_pdb_input: str, binder_contig: str,  motif_id: str):
    # 1. Extract only the alpha-carbons from the holo complexes
    coords_motif_design = extract_design_motif_coords(path_pdb_design, binder_contig, desired_motif_chain_id= motif_id)
    coords_motif_input = extract_input_motif_coords(path_pdb_input, binder_contig, desired_motif_chain_id= motif_id)
    aligned_design_coords, applied_transformation = superimpose(coords_motif_input, coords_motif_design)
    calculated_rmsd = rmsd(coords_motif_input, aligned_design_coords)
    return calculated_rmsd

In [0]:
import os
import sys
import biotite.structure as struc
from biotite.structure import superimpose, rmsd

if motif_scaffold:
    
    list_rmsd_motif_desired = []
    list_rmsd_motif = []
    list_design_paths = []
    list_plddt_binder = []
    for index, row in df_mpnn.iterrows():
    
        # 1. Extract path of the pdb design
        path_pdb_design = path_pdbs + f"/design{row['design']}_n{row['n']}.pdb"

        # 2.5: Compute Binder PLDDT:
        plddt_binder = compute_chain_plddt(path_pdb_design, chain_id= "B")
    
        # 3. Extract contigs as a string
        contigs_str = str(row['contig'])
        print("Contigs_str: ", contigs_str)
        binder_contig = extract_binder_contig(contigs_str)
        print("Binder_Contig: ", binder_contig)

        # 4. Extract coordinates of full motif in the design and input & calculate rmsd between them
        rmsd_motif = calculate_rmsd_aligned_coords(path_pdb_design, path_input_pdb, binder_contig, motif_id= "")

        # 5. Extract coordinates of desired motif in the design and input
        if desired_motif_chain_id in binder_contig:
            rmsd_motif_desired = calculate_rmsd_aligned_coords(path_pdb_design, path_input_pdb, binder_contig, motif_id= desired_motif_chain_id)
            list_rmsd_motif_desired.append(rmsd_motif_desired)
    
        # 7. Append RMSD & Binder-Specific PLDDT & Design PDB Path to respective lists
        list_rmsd_motif.append(rmsd_motif)
        list_plddt_binder.append(plddt_binder)
        list_design_paths.append(path_pdb_design)

    # 8. Insert RMSD & Binder-Specific PLDDT & Design PDB Path to respective columns
    df_mpnn["plddt_binder"] = list_plddt_binder
    # 9. Special Condition for when no desired motif is present
    if (len(list_rmsd_motif_desired) == 0):
        df_mpnn["rmsd_motif_desired"] = [-1.0] * len(list_plddt_binder)
    else:
        df_mpnn["rmsd_motif_desired"] = list_rmsd_motif_desired
    # 10. Insert RMSD between motif in design and input & design pdb path
    df_mpnn["rmsd_motif"] = list_rmsd_motif
    df_mpnn['design_pdb_path'] = list_design_paths
    # 11. Save to csv
    df_mpnn.to_csv(path_mpnn_csv)

In [0]:
df_mpnn